In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import KFold
from sklearn.model_selection import TimeSeriesSplit
from matplotlib import pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn 

from src.config import TARGET, TIME_COL, UID_COL, SEED, DAGSHUB_USERNAME,DAGSHUB_REPO 
from src.data.nn_preprocessing import prepare_nn_fold_data, build_vectorized_user_sequences
from src.data.nn_dataset import FraudMultiModalDataset
from src.models.modules.fused import FraudMultiModalModel
from src.models.nn_trainer import train_and_validate_multimodal

from google.colab import drive
import os
import mlflow
import mlflow.pytorch
import dagshub
from dotenv import load_dotenv

In [2]:
%pip install -q mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 111.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.5/228.5 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Mount Google Drive into the remote Colab container
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/IEEE-fraud-detection"
os.makedirs(SAVE_DIR, exist_ok=True)

# 2. Load credentials directly from Google Drive .env file
ENV_PATH = os.path.join(SAVE_DIR, ".env")
load_dotenv(ENV_PATH)

dagshub.init(
    repo_owner=DAGSHUB_USERNAME,
    repo_name=DAGSHUB_REPO,
    mlflow=True
)

mlflow.set_experiment("IEEE-fraud-multimodal-nn")
print(f"MLflow tracking set to DagsHub")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=74ce4605-07e0-4169-a894-6c66ad1d5c36&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=45f3324ff41eba566e652dd10c336dff8b35114492c873bb3bb6e82d91d8801d




Accessing as anamish05

Initialized MLflow to track repo "anamish05/ieee-fraud-detection"

Repository anamish05/ieee-fraud-detection initialized!

2026/10/01 11:12:56 INFO mlflow.tracking.fluent: Experiment with name 'IEEE-fraud-multimodal-nn' does not exist. Creating a new experiment.


MLflow tracking set to DagsHub


In [7]:
from pathlib import Path

# Path inside mounted Google Drive
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/IEEE-fraud-detection")
PROCESSED_PATH = DRIVE_PROJECT_ROOT / "data/processed/features_static_after_split_TE.parquet"

# Verify file exists before loading
assert PROCESSED_PATH.exists(), f"File not found at: {PROCESSED_PATH}"

df = pd.read_parquet(PROCESSED_PATH)
print(f"Loaded engineered dataset: {df.shape}")

Loaded engineered dataset: (590540, 327)


In [ ]:
# split as in the part 3 LGBM
n_samples = len(df)
split_idx = int(n_samples * 0.8)

idx_tr_time = np.arange(0, split_idx)
idx_val_time = np.arange(split_idx, n_samples)

Prepare data for NN - impute NaNs, scale features

In [10]:
TARGET='isFraud'
te_features = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M4']
drop_cols = [
    TARGET, 
    #'TransactionDT', # will need them for sequence building next step
    'Pseudo_UID',
    'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26' # sparse IP/proxy with >95% NaN
] + te_features  # Drops raw ProductCD, card4, etc.

features = [c for c in df.columns if c not in drop_cols]

cat_cols=['card1', 'card2', 'card3', 'card5',
                        'addr1', 'addr2',
                        'DeviceInfo',
                        'id_30', 'id_31', 'id_33', 'id_13', 'id_14', 'id_17', 'id_19', 'id_20', 'id_32', 'id_34'] 

bin_cols=['M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9', 
          'id_12', 'id_15', 'id_16', 'id_27', 'id_28', 'id_29', 'id_35', 'id_36', 'id_37', 'id_38', 
          'DeviceType']

block_bin_flags = [c for c in ['block_2_present', 'block_3_present', 'block_4_present'] if c in df.columns]

te_cols=[c for c in features if c.endswith('_target_enc')]

freq_cols = [c for c in df.columns if c.endswith('_freq')] # apply log1p before scaling

special_excluded = set(drop_cols + cat_cols + bin_cols + block_bin_flags + te_cols + freq_cols + ['uid_dt_diff'])
numeric_cols = [c for c in df.columns if c not in special_excluded]

In [12]:
X_train_raw = df.iloc[idx_tr_time]
X_val_raw   = df.iloc[idx_val_time]
y_train     = df.iloc[idx_tr_time][TARGET].values.astype(np.float32)
y_val       = df.iloc[idx_val_time][TARGET].values.astype(np.float32)

In [13]:
# 2. Run preprocessing
X_tr_cont, X_va_cont, X_tr_cat, X_va_cat, emb_szs, cont_names = prepare_nn_fold_data(
    X_train_raw=X_train_raw,
    X_val_raw=X_val_raw,
    numeric_cols=numeric_cols,
    freq_cols=freq_cols,
    cat_cols=cat_cols,
    bin_cols=bin_cols,
    block_bin_flags=block_bin_flags,
    te_cols=te_cols,
    min_freq=50,
    scaler_type='robust'
)

/tmp/ipykernel_2868/1327183276.py:62: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  X_tr['id_15'] = X_tr['id_15'].replace('Unknown', np.nan)
/tmp/ipykernel_2868/1327183276.py:63: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  X_va['id_15'] = X_va['id_15'].replace('Unknown', np.nan)
/tmp/ipykernel_2868/1327183276.py:147: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downca

In [14]:
# 3. Assert zero NaNs and verify shapes
assert not np.isnan(X_tr_cont).any(), "NaN found in X_tr_cont!"
assert not np.isnan(X_va_cont).any(), "NaN found in X_va_cont!"
assert not np.isnan(X_tr_cat).any(),  "NaN found in X_tr_cat!"
assert not np.isnan(X_va_cat).any(),  "NaN found in X_va_cat!"

print(f"X_tr_cont shape: {X_tr_cont.shape} (dtype: {X_tr_cont.dtype})")
print(f"X_tr_cat  shape: {X_tr_cat.shape}  (dtype: {X_tr_cat.dtype})")
print(f"Total categorical embedding tables: {len(emb_szs)}")

X_tr_cont shape: (472432, 318) (dtype: float32)
X_tr_cat  shape: (472432, 17)  (dtype: int64)
Total categorical embedding tables: 17


Part 2. Sequential track.  
Iterating row-by-row or using Python loops over ~590K transactions to construct sliding transaction histories will crash memory or take hours.To build sequences efficiently, we can use a NumPy index stride map:  
- Sort rows by (pseudo_UID, TransactionDT).  
- Compute each transaction's chronological rank within its user group using cumcount().  
- Map every transaction index backwards by up to $L$ steps within the same user boundary using pure array operations.  
- Gather sequential feature vectors into a 3D float32 tensor $\mathbf{X}_{\text{seq}} \in \mathbb{R}^{N \times L \times D_{\text{seq}}}$ and a boolean padding mask $\mathbf{M}_{\text{pad}} \in \{0, 1\}^{N \times L}$ (where True denotes padded/masked time steps for PyTorch's src_key_padding_mask).

Key Performance Benefits:Execution Time: Processes all ~590K transactions in ~1.5 to 2.5 seconds (compared to >30 minutes with .groupby().apply()).Memory Footprint: For $N=590{,}000$, $L=15$, and $D_{\text{seq}}=5$, X_seq consumes approximately 177 MB of RAM, fitting comfortably in memory.PyTorch Integration: seq_padding_mask matches PyTorch's native nn.TransformerEncoder(..., src_key_padding_mask=mask) directly, where True flags padded tokens to receive $-\infty$ attention weights.

In [ ]:
# 1. Define 5-8 temporal features per sequence step
seq_features = [
    'TransactionAmt_log',       # np.log1p(df['TransactionAmt'])
    'uid_dt_diff_log',          # Elapsed seconds since last transaction
    'addr1_freq',               # Rarity of billing address
    'C1',                       # Phone/email verification frequency
    'C2'                        # Cardholder verification counter
]

df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'].clip(lower=0)).astype(np.float32)

# handle NaT first transactions for uid_dt_diff
MAX_ELAPSED_LOG = 365 * 24 * 3600
df['uid_dt_diff_log']=np.log1p(df['uid_dt_diff'].dt.total_seconds().fillna(MAX_ELAPSED_LOG).clip(lower=0))

# 2. Build 3D sequences (capped at last 15 transactions per user)
X_seq, seq_padding_mask = build_vectorized_user_sequences(
    df=df,
    uid_col=UID_COL,
    time_col=TIME_COL,
    seq_feature_cols=seq_features,
    max_len=15
)

print(f"X_seq shape:        {X_seq.shape} ({X_seq.nbytes / (1024**2):.1f} MB)")
print(f"padding_mask shape: {seq_padding_mask.shape}")

X_seq shape:        (590540, 15, 5) (169.0 MB)
padding_mask shape: (590540, 15)


In [17]:
X_tr_seq = X_seq[idx_tr_time]
X_va_seq = X_seq[idx_val_time]

mask_tr  = seq_padding_mask[idx_tr_time]
mask_va  = seq_padding_mask[idx_val_time]

A standard Transformer expects a continuous hidden dimension ($d_{\text{model}}$), positional awareness, and attention masking. The sequence pipeline performs these operations:Linear Projection: Projects the raw $D_{\text{seq}}$ features (e.g., dimension 5) up to the model dimension $d_{\text{model}}$ (e.g., 64 or 128).Positional Encoding: Adds learned or sinusoidal positional vectors so the network understands the chronological order of transactions ($t-14, \dots, t-1, t$).Transformer Encoder: Self-attention layers compute pairwise correlations between past transactions, masked by src_key_padding_mask so padded dummy steps are assigned $-\infty$ attention weights.Sequence Pooling: Aggregates the $L$ step vectors into a single summary vector $\mathbf{h}_{\text{seq}} \in \mathbb{R}^{d_{\text{model}}}$ representing the user's recent behavioral trajectory. The most effective pooling strategy here is extracting the final non-padded step (the current transaction at index $t$), optionally concatenated with a masked mean over the past sequence.

Track 3 - device-text   
Device Info, id_30 and  id_31. - need cuda

In [19]:
df['device_text'] = (
    "device: " + df['DeviceInfo'].astype(str).replace("nan","missing") + 
    " os: " + df['id_30'].astype(str).replace("nan","missing") + 
    " browser: " + df['id_31'].astype(str).replace("nan","missing")
)

# extract only unique text strings and then map them back to the df
unique_texts=df['device_text'].unique().tolist()
print(f"Total rows: {len(df):,}, Unique device strings: {len(unique_texts):,}")

Total rows: 590,540, Unique device strings: 6,419


In [20]:
from sentence_transformers import SentenceTransformer

# 'all-MiniLM-L6-v2' is fast, lightweight (~80MB), and outputs 384-dim vectors
text_encoder = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

unique_embeddings = text_encoder.encode(
    unique_texts, 
    batch_size=256, 
    show_progress_bar=True,
    convert_to_numpy=True
)

# 4. Map unique embeddings back to all rows via integer lookup
text_to_idx = {text: idx for idx, text in enumerate(unique_texts)}
row_indices = df['device_text'].map(text_to_idx).to_numpy()

# 5. Resulting matrix: shape (N, 384), dtype float32
X_text_emb = unique_embeddings[row_indices].astype(np.float32)

# 6. Slice to train and validation sets
X_tr_text = X_text_emb[idx_tr_time]
X_va_text = X_text_emb[idx_val_time]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

Assemble the Multi-Modal PyTorch Dataset & DataLoader

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    FraudMultiModalDataset(X_tr_cont, X_tr_cat, X_tr_seq, mask_tr, X_tr_text, y_train),
    batch_size=2048, shuffle=True, drop_last=True
) # big batch cause of imbalance, shuffle to break chronological blocks and prevent gradient drifts, drop last to avoid tiny branch
val_loader = DataLoader(
    FraudMultiModalDataset(X_va_cont, X_va_cat, X_va_seq, mask_va, X_va_text, y_val),
    batch_size=4096, shuffle=False)  # deterministic approach, each epoch same sequence

Join All 3 Branches in One Model Architecture

Training and validation loop

In [ ]:
n_neg, n_pos =(y_train==0).sum(), (y_train==1).sum()
pos_weight_val=float(n_neg/n_pos)**(0.5)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pos_weight = torch.tensor([pos_weight_val], dtype=torch.float32, device=device)

# BCE with logits incorporates sigmoid internally for numerical stability
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f"Calculated pos_weight: {pos_weight_val:.2f} on device: {device}")

Calculated pos_weight: 5.24 on device: cuda


In [25]:
print("X_tr_cont NaNs:", np.isnan(X_tr_cont).sum())
print("X_va_cont NaNs:", np.isnan(X_va_cont).sum())
print("X_tr_seq NaNs: ", np.isnan(X_tr_seq).sum())
print("X_va_seq NaNs: ", np.isnan(X_va_seq).sum())
print("X_tr_text NaNs:", np.isnan(X_tr_text).sum())
print("X_va_text NaNs:", np.isnan(X_va_text).sum())

X_tr_cont NaNs: 0
X_va_cont NaNs: 0
X_tr_seq NaNs:  0
X_va_seq NaNs:  0
X_tr_text NaNs: 0
X_va_text NaNs: 0


Instantiate Model, Optimizer, and Run

In [26]:
# 1. Instantiate the Model
model = FraudMultiModalModel(
    emb_szs=emb_szs,
    n_cont=X_tr_cont.shape[1],
    d_seq=X_tr_seq.shape[-1],      # Number of sequence features (e.g., 5)
    d_text=X_tr_text.shape[-1],     # MiniLM text dimension (384)
    cat_dropout=0.1
).to(device)

# different lr for transformer
transformer_params = list(model.seq_transformer.parameters())
other_params = [p for p in model.parameters() if not any(p is tp for tp in transformer_params)]

# 2. AdamW Optimizer with mild weight decay for tabular regularization
optimizer = torch.optim.AdamW([
    {'params':other_params, 'lr':1e-3, 'weight_decay':1e-4},
    {'params': transformer_params, 'lr':3e-4, 'weight_decay':1e-3}
])

# 3. Learning rate decay when validation metric plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=1,
    min_lr=1e-5
)

# 4. Parameters to log in MLflow
params = {
    "architecture": "FraudMultiModalModel",
    "cat_dropout": 0.10,
    "pos_weight": pos_weight,
    "lr_static": 1e-3,
    "lr_transformer": 3e-4,
    "batch_size": train_loader.batch_size,
    "d_seq": X_tr_seq.shape[-1]
}

# 5. Run Training
trained_model, history, oof_val_preds = train_and_validate_multimodal(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=15,
    patience=3,
    device=device,
    save_dir=SAVE_DIR,
    run_name="multimodal_seq128_01",
    params_dict=params
)

Epoch 01/15 | Train Loss: 0.3673 | Val Loss: 0.3286 | Val ROC-AUC: 0.86682
  --> Checkpoint saved to Google Drive (AUC: 0.86682)
Epoch 02/15 | Train Loss: 0.2833 | Val Loss: 0.3228 | Val ROC-AUC: 0.87062
  --> Checkpoint saved to Google Drive (AUC: 0.87062)
Epoch 03/15 | Train Loss: 0.2583 | Val Loss: 0.3192 | Val ROC-AUC: 0.87764
  --> Checkpoint saved to Google Drive (AUC: 0.87764)
Epoch 04/15 | Train Loss: 0.2418 | Val Loss: 0.3274 | Val ROC-AUC: 0.88623
  --> Checkpoint saved to Google Drive (AUC: 0.88623)
Epoch 05/15 | Train Loss: 0.2282 | Val Loss: 0.3194 | Val ROC-AUC: 0.88218
Epoch 06/15 | Train Loss: 0.2170 | Val Loss: 0.3317 | Val ROC-AUC: 0.88456
Epoch 07/15 | Train Loss: 0.2005 | Val Loss: 0.3261 | Val ROC-AUC: 0.88662
  --> Checkpoint saved to Google Drive (AUC: 0.88662)
Epoch 08/15 | Train Loss: 0.1923 | Val Loss: 0.3257 | Val ROC-AUC: 0.88687
  --> Checkpoint saved to Google Drive (AUC: 0.88687)
Epoch 09/15 | Train Loss: 0.1892 | Val Loss: 0.3339 | Val ROC-AUC: 0.88876
 

2026/10/01 11:21:14 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/10/01 11:21:14 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/10/01 11:21:23 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version la

Training complete. Dagshub MLflow run logged: 4075eac73e8245998b8c1ca1eb1655b5
🏃 View run multimodal_seq128_01 at: https://dagshub.com/anamish05/ieee-fraud-detection.mlflow/#/experiments/1/runs/4075eac73e8245998b8c1ca1eb1655b5
🧪 View experiment at: https://dagshub.com/anamish05/ieee-fraud-detection.mlflow/#/experiments/1
